# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [2]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [3]:
# Task 1
# YOUR CODE HERE

Types = ['Entire home/apt','Private room']

df_capROOM = df_cap[df_cap['room_type'].isin(Types)]
df_EntireRoom = df_capROOM.loc[df_capROOM['room_type'] == Types[0]]
df_PrivateRoom = df_capROOM.loc[df_capROOM['room_type'] == Types[1]]

fig = px.histogram(
    data_frame=df_capROOM, 
    x='price',
    nbins=50,                              # 50 bins: enough granularity to see shape
    color='room_type',
    barmode='overlay',  
    opacity=0.6,
    labels={'price': 'Nightly Price (£)', 'count': 'Number of Listings'},
    height=500
)


# Add median line
median_price1 = df_EntireRoom['price'].median()
median_price2 = df_PrivateRoom['price'].median()

fig.add_vline(x=median_price1, line_dash='dash', line_color='#E63946', line_width=2,
              annotation=dict(
                  text=f'<b>Median: £{median_price1:.0f}',
                  font=dict(color='#E63946', size=12),
                  xanchor='left',
                  yanchor='top',
                  xshift=8,   # pixels to the right of the line
              ))

fig.add_vline(x=median_price2, line_dash='dash', line_color='#E63946', line_width=2,
              annotation=dict(
                  text=f'<b>Median: £{median_price2:.0f}',
                  font=dict(color='#E63946', size=12),
                  xanchor='left',
                  yanchor='top',
                  xshift=8,   # pixels to the right of the line
              ))

fig.update_layout(
    title=f'London Airbnb prices are heavily right-skewed — most listings are under £150, but outliers reach £{p95:.0f}+',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='Number of Listings'),
    xaxis=dict(showgrid=False),
    margin=dict(l=60, r=40, t=65, b=40)
)
fig.show()

## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [15]:
# Task 2
# YOUR CODE HERE

df_cap.replace({"number_of_reviews":0}, np.nan, inplace = True)
df_cap.dropna(subset=["number_of_reviews"], axis=0, inplace = True)
med_order = df_cap.groupby('neighbourhood')['reviews_per_month'].median().sort_values(ascending=False).index.tolist()
df_cap1 = df_cap.dropna(subset=['reviews_per_month']).copy()
print(med_order)
df_cap1 = df_cap1.sort_values(['reviews_per_month','neighbourhood'], ascending=False)

fig = px.box(
    data_frame=df_cap1,
    y='reviews_per_month', x='neighbourhood',
    color='neighbourhood',
    # Meaningful colour: entire home (most expensive) gets the standout colour
    color_discrete_sequence=['#DDDDDD']*len(med_order), 
    notched=True,                  # notch shows confidence interval around median
    points='outliers',             # show outliers as individual points
    category_orders={'neighbourhood': med_order},
    labels={'price': 'Nightly Price (£)', 'room_type': ''},
    #category_orders={'room_type': ['Entire home/apt', 'Private room', 'Shared room']}, 
    height=500
)

expensive = med_order[:2]
for trace in fig.data:
    if trace.name in expensive:
        trace.line.color = 'rgba(230,57,70,1)'
        trace.fillcolor = 'rgba(230,57,70,0.25)' # lighter shade of the line color using a = opacity

fig.update_layout(
    title='Westminster and Kensington and Chelsea are the places with ',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#EEEEEE', title='Nightly Price (£)'),
    yaxis=dict(showgrid=False),
    showlegend=False,
    margin=dict(l=60, r=40, t=55, b=40)
)
fig.show()


['Tower Hamlets', 'Hackney', 'Camden', 'Southwark', 'Westminster', 'Hammersmith', 'Kensington and Chelsea', 'Lambeth', 'Islington', 'Wandsworth']
